# BTC Orderbook LSTM — Microstructure Direction Prediction

**Best parameters from XGBoost sweep:** `--threshold 50 --horizon 8` (2-minute horizon)

**Architecture:** LSTM over a sliding window of the last N bars, predicting UP/DOWN/FLAT at horizon=8 bars ahead.

**Same feature pipeline and bias rules as `btc_signal_analysis.py` — no lookahead.**

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Mount Google Drive (Cell 2) and set `DATA_DIR` to your CSV/CSV.GZ folder
3. Run all cells in order

## Key design decisions
- **Sequence length = 32 bars (8 minutes):** captures sustained imbalance patterns
- **Chronological train/val/test split** — no shuffling, no future leakage
- **Class weights** to handle FLAT dominance (~55%)
- **Same backtest** as XGBoost: win rate, avg P&L per trade, after-fee comparison

In [ ]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'scikit-learn', 'pandas', 'numpy', 'matplotlib'], check=True)

import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/content/drive/MyDrive/btc_data'  # folder with your CSV/CSV.GZ files
OUTPUT_DIR = '/content/drive/MyDrive/btc_lstm'  # where to save model and charts
THRESHOLD  = 50.0    # USD — same as best XGBoost run
HORIZON    = 8       # bars ahead = 2 minutes
SEQ_LEN    = 32      # bars of history fed to LSTM = 8 minutes
BATCH_SIZE = 2048
EPOCHS     = 40
LR         = 1e-3
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT    = 0.3
TRAIN_FRAC = 0.70    # chronological split
VAL_FRAC   = 0.15    # validation (early stopping)
# TEST_FRAC  = 0.15  # remainder used for backtest
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Feature pipeline — exact copy from btc_signal_analysis.py ────
#
# All feature definitions and bias rules are preserved.
# Only the columns needed are read from CSV (memory optimisation).

SEPARATOR = ';'
WINDOW_SEC = 15

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]

ALL_FEATURES = [
    # Original 25
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    # Sub-bar (if available)
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
]


def load_files(files):
    all_needed = COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP) | set(COLS_NEEDED_LIQ_TOTAL)
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
            print(f'  Loaded {os.path.basename(path):45s}  {len(df):>7,} rows  ({len(usecols)} cols)')
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    return pd.concat(frames, ignore_index=True)


def prepare(df, threshold, horizon):
    """Exact feature pipeline from btc_signal_analysis.py. No lookahead bias."""
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    next_close   = df['close_price'].shift(-horizon)
    delta        = next_close - df['close_price']
    df['target'] = np.where(delta > threshold, 1, np.where(delta < -threshold, -1, 0))
    df['delta_usd'] = delta

    eps = 1e-9

    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(4, min_periods=2).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(8, min_periods=4).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(4, min_periods=2).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(8, min_periods=4).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    # Sub-bar features
    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    # Book shape
    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    # Time features
    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    # Volatility
    df['volatility'] = df['close_price'].diff().abs().rolling(16, min_periods=4).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(5760, min_periods=96).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    df = df.iloc[:-horizon].copy()
    return df

print('Feature pipeline loaded.')

In [ ]:
# ── Cell 4: Load and prepare data ────────────────────────────────────────
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')) +
               glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
print(f'Found {len(files)} files')

raw = load_files(files)
print(f'\nTotal rows loaded: {len(raw):,}')

df = prepare(raw, THRESHOLD, HORIZON)
del raw  # free memory

# Select available features
features = [f for f in ALL_FEATURES if f in df.columns and df[f].notna().sum() > 100]
print(f'\nFeatures available: {len(features)}/35')
print(f'Date range: {df["dt"].iloc[0]} → {df["dt"].iloc[-1]}')
print(f'Total rows after prepare: {len(df):,}')

# Class distribution
vc = df['target'].value_counts().sort_index()
for k, v in vc.items():
    label = {-1: 'DOWN', 0: 'FLAT', 1: 'UP'}[k]
    print(f'  {label}: {v:,} ({v/len(df)*100:.1f}%)')

In [ ]:
# ── Cell 5: Build sequences ───────────────────────────────────────────────
#
# For each row N, the input is [bar_{N-SEQ_LEN+1}, ..., bar_N] — SEQ_LEN bars
# of feature history. The label is target[N].
#
# Bias guarantee: sequence[i] for position N uses only rows 0..N.
# Same truncation test as btc_signal_analysis.py verifies this.
#
# Rows with ANY NaN feature are excluded (model can't handle NaN).
# This removes warm-up rows at the start and sub-bar NaN rows from old files.

clean = df[features + ['target', 'close_price', 'dt']].dropna().reset_index(drop=True)
print(f'Clean rows (no NaN in any feature): {len(clean):,}  '
      f'({len(clean)/len(df)*100:.1f}% of prepared data)')

X_all = clean[features].values.astype(np.float32)
y_all = clean['target'].values
price_all = clean['close_price'].values
dt_all    = clean['dt'].values

# Scale features BEFORE building sequences — scaler fitted on train only
n         = len(clean)
train_end = int(n * TRAIN_FRAC)
val_end   = int(n * (TRAIN_FRAC + VAL_FRAC))

scaler = StandardScaler()
X_all[:train_end] = scaler.fit_transform(X_all[:train_end])
X_all[train_end:] = scaler.transform(X_all[train_end:])
print(f'Scaler fitted on {train_end:,} training rows')

# Build sliding window sequences
def make_sequences(X, y, seq_len):
    """Return (sequences, labels) arrays with shape (N-seq_len, seq_len, F) and (N-seq_len,)."""
    N, F = X.shape
    seqs   = np.lib.stride_tricks.sliding_window_view(X, (seq_len, F)).reshape(-1, seq_len, F)
    labels = y[seq_len - 1:]  # label is at the LAST bar of each window
    return seqs.astype(np.float32), labels

# Build sequences per split to avoid cross-split contamination
X_train_seq, y_train = make_sequences(X_all[:train_end], y_all[:train_end], SEQ_LEN)
X_val_seq,   y_val   = make_sequences(X_all[train_end:val_end], y_all[train_end:val_end], SEQ_LEN)
X_test_seq,  y_test  = make_sequences(X_all[val_end:], y_all[val_end:], SEQ_LEN)
price_test            = price_all[val_end + SEQ_LEN - 1:]
dt_test               = dt_all[val_end + SEQ_LEN - 1:]

print(f'\nSequences built (seq_len={SEQ_LEN} bars = {SEQ_LEN*WINDOW_SEC}s):')  
print(f'  Train: {len(X_train_seq):,}  Val: {len(X_val_seq):,}  Test: {len(X_test_seq):,}')
print(f'  Input shape: {X_train_seq.shape}  (samples, seq_len, features)')

In [ ]:
# ── Cell 6: Dataset and DataLoader ───────────────────────────────────────

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        # Map -1→0, 0→1, 1→2 for CrossEntropyLoss
        self.y = torch.tensor(y + 1, dtype=torch.long)
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_ds = SeqDataset(X_train_seq, y_train)
val_ds   = SeqDataset(X_val_seq,   y_val)
test_ds  = SeqDataset(X_test_seq,  y_test)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Class weights to handle FLAT dominance
cw = compute_class_weight('balanced', classes=np.array([-1, 0, 1]), y=y_train)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)
print(f'Class weights: DOWN={cw[0]:.2f}  FLAT={cw[1]:.2f}  UP={cw[2]:.2f}')

In [ ]:
# ── Cell 7: LSTM model ────────────────────────────────────────────────────

class BtcLSTM(nn.Module):
    """
    Stacked LSTM with:
      - Layer norm on input (stabilises training on heterogeneous features)
      - Dropout between layers
      - Linear classifier on final hidden state
    """
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, num_classes=3):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,  # causal: can only see past bars
        )
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, features)
        x   = self.input_norm(x)
        out, _ = self.lstm(x)
        # Use only the last timestep's hidden state
        last = out[:, -1, :]  # (batch, hidden)
        return self.fc(self.dropout(last))  # (batch, 3)


model = BtcLSTM(
    input_dim=len(features),
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

In [ ]:
# ── Cell 8: Training ──────────────────────────────────────────────────────

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_loss = float('inf')
best_state    = None
patience      = 8
wait          = 0

train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train
    model.train()
    running_loss = 0.0
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * len(X_b)
    train_loss = running_loss / len(train_ds)

    # ── Validate
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in val_dl:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            val_loss += criterion(logits, y_b).item() * len(X_b)
            preds    = logits.argmax(1)
            correct += (preds == y_b).sum().item()
            total   += len(y_b)
    val_loss /= len(val_ds)
    val_acc   = correct / total
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait          = 0
        flag = '  ← best'
    else:
        wait += 1
        flag  = f'  (patience {wait}/{patience})'

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  '
              f'val_acc={val_acc*100:.2f}%  lr={lr_now:.2e}{flag}')

    if wait >= patience:
        print(f'\nEarly stopping at epoch {epoch} (patience={patience})')
        break

# Restore best weights
model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# ── Cell 9: Evaluation and backtest ──────────────────────────────────────

le_inv = {0: -1, 1: 0, 2: 1}  # class index → target value

model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_dl:
        logits = model(X_b.to(device)).cpu()
        all_logits.append(logits)
        all_labels.append(y_b)

logits_all = torch.cat(all_logits)
proba_all  = torch.softmax(logits_all, dim=1).numpy()   # (N, 3)
pred_enc   = logits_all.argmax(1).numpy()               # 0/1/2
preds      = np.array([le_inv[p] for p in pred_enc])    # -1/0/1
confidence = proba_all.max(axis=1)                      # max class prob

y_test_orig = np.array([le_inv[l.item()] for l in all_labels[0].unbind()] +
                        [le_inv[l.item()] for lb in all_labels[1:] for l in lb.unbind()])

# Accuracy
dummy_acc = max(np.bincount(pred_enc + 1) / len(pred_enc))
acc       = (preds == y_test_orig).mean()
dummy_acc = max(np.bincount([{-1:0,0:1,1:2}[v] for v in y_test_orig]) / len(y_test_orig))
print(f'Test accuracy : {acc*100:.2f}%')
print(f'Dummy baseline: {dummy_acc*100:.2f}%')
print(f'Lift          : {(acc - dummy_acc)*100:+.2f}%')

# Backtest: same method as btc_signal_analysis.py
# price_test[i] = close price at bar i of the test set
# trade P&L = predicted direction × next bar price move
price_moves = np.diff(price_test, prepend=price_test[0])
# For bar N: if we enter at close_price[N] and exit at close_price[N+1],
# P&L = preds[N] * (close_price[N+1] - close_price[N])
# We use the actual next-bar price move (price_moves[N+1])
# Note: last bar has no exit — drop it
trade_pnl = np.where(preds[:-1] ==  1,  np.diff(price_test),
            np.where(preds[:-1] == -1, -np.diff(price_test), 0.0))

bnh_pnl    = price_test[-1] - price_test[0]
strat_pnl  = trade_pnl.sum()
n_trades   = (preds[:-1] != 0).sum()
n_long     = (preds[:-1] ==  1).sum()
n_short    = (preds[:-1] == -1).sum()
trades_pnl = trade_pnl[preds[:-1] != 0]
win_rate   = (trades_pnl > 0).mean() if len(trades_pnl) > 0 else 0
avg_win    = trades_pnl[trades_pnl > 0].mean() if (trades_pnl > 0).any() else 0
avg_loss   = trades_pnl[trades_pnl < 0].mean() if (trades_pnl < 0).any() else 0

avg_price  = price_test.mean()
maker_fee  = avg_price * 0.0004  # 0.02% per side
taker_fee  = avg_price * 0.0010  # 0.05% per side

print(f'\n{'─'*60}')
print(f'BACKTEST (test set, zero fees)')
print(f'{'─'*60}')
print(f'Test period  : {pd.Timestamp(dt_test[0])} → {pd.Timestamp(dt_test[-1])}')
print(f'BTC price    : ${price_test[0]:,.0f} → ${price_test[-1]:,.0f}  (Δ${price_test[-1]-price_test[0]:+,.0f})')
print(f'Total signals: {n_trades:,}  ({n_trades/len(preds)*100:.1f}% of bars)')
print(f'Long / Short : {n_long:,} / {n_short:,}')
print(f'Strategy P&L : ${strat_pnl:+,.2f}  (vs B&H ${bnh_pnl:+,.2f})')
print(f'Win rate     : {win_rate*100:.2f}%  over {n_trades:,} trades')
print(f'Avg win/trade: ${avg_win:.2f}  |  Avg loss: ${avg_loss:.2f}')
print(f'Expected P&L/trade: ${win_rate*avg_win + (1-win_rate)*avg_loss:.2f}')
print(f'Maker fee/trade  : ${maker_fee:.2f}')
print(f'After maker fees : ${strat_pnl - n_trades*maker_fee:+,.2f}')
print(f'After taker fees : ${strat_pnl - n_trades*taker_fee:+,.2f}')

# Confidence threshold sweep
print(f'\nConfidence threshold sweep (maker fee = ${maker_fee:.0f}/trade):')
print(f'  {"Threshold":>10}  {"Trades":>7}  {"WinRate":>8}  {"AvgP&L":>8}  {"After maker":>12}')
for thresh in [0.33, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    mask   = confidence[:-1] >= thresh
    fp     = np.where(mask & (preds[:-1] ==  1),  np.diff(price_test),
             np.where(mask & (preds[:-1] == -1), -np.diff(price_test), 0.0))
    nt     = mask.sum()
    if nt == 0:
        print(f'  {thresh:>10.0%}  {0:>7,}  {"":>8}  {"":>8}  {"":>12}')
        continue
    tp     = fp[mask]
    wr     = (tp > 0).mean()
    avg    = fp.sum() / nt
    after  = fp.sum() - nt * maker_fee
    ok     = '✓' if after > 0 else '✗'
    print(f'  {thresh:>10.0%}  {nt:>7,}  {wr*100:>7.2f}%  {avg:>+8.2f}  {after:>+12,.0f}  {ok}')

# Binomial test
try:
    from scipy.stats import binomtest
    n_wins = int(n_trades * win_rate)
    p = binomtest(n_wins, n_trades, 0.5, alternative='greater').pvalue
    print(f'\nBinomial test: win rate {win_rate*100:.2f}% over {n_trades:,} trades → p={p:.4f} '
          f'{"✓ significant" if p < 0.05 else "✗ not significant"}')
except:
    pass

In [ ]:
# ── Cell 10: Plots ────────────────────────────────────────────────────────

DARK, PANEL, GRID = '#0d1117', '#161b22', '#21262d'
TEXT, UP, DOWN, ACC = '#e6edf3', '#3fb950', '#f85149', '#58a6ff'
GOLD = '#f0883e'

fig = plt.figure(figsize=(18, 14), facecolor=DARK)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.40, wspace=0.30,
                        left=0.07, right=0.97, top=0.93, bottom=0.07)

def style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.set_title(title, color=TEXT, fontsize=9, fontweight='bold', pad=6)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(color=GRID, linewidth=0.5, alpha=0.7)
    ax.yaxis.label.set_color(TEXT)
    ax.xaxis.label.set_color(TEXT)

# 1. Training curves
ax1 = fig.add_subplot(gs[0, 0])
ep = range(1, len(train_losses) + 1)
ax1.plot(ep, train_losses, color=ACC, label='Train loss')
ax1.plot(ep, val_losses,   color=GOLD, label='Val loss')
ax1.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
style(ax1, 'Training / Validation Loss')

ax1b = fig.add_subplot(gs[0, 1])
ax1b.plot(ep, [v*100 for v in val_accs], color=UP)
ax1b.axhline(dummy_acc*100, color=GRID, linestyle='--', label='Dummy baseline')
ax1b.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax1b.set_xlabel('Epoch'); ax1b.set_ylabel('Accuracy (%)')
style(ax1b, 'Validation Accuracy')

# 2. Cumulative P&L
ax2 = fig.add_subplot(gs[1, :])
cum_strat = np.cumsum(trade_pnl)
cum_bnh   = price_test[1:] - price_test[0]
x_idx     = np.arange(len(cum_strat))
ax2.plot(x_idx, cum_strat, color=ACC, linewidth=1.2, label=f'LSTM (zero fees)')
ax2.plot(x_idx, cum_bnh,   color=GOLD, linewidth=1.0, linestyle='--', label='Buy & Hold')
ax2.axhline(0, color=GRID, linewidth=0.8)
ax2.fill_between(x_idx, cum_strat, 0, where=cum_strat >= 0, alpha=0.12, color=UP)
ax2.fill_between(x_idx, cum_strat, 0, where=cum_strat <  0, alpha=0.12, color=DOWN)
ax2.set_ylabel('Cumulative P&L (USD)')
ax2.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=9)
ax2.set_title(f'Backtest: LSTM ${strat_pnl:+,.0f} vs B&H ${bnh_pnl:+,.0f}  |  '
              f'WR {win_rate*100:.2f}%  |  {n_trades:,} trades',
              color=TEXT, fontsize=10, fontweight='bold')
for sp in ax2.spines.values(): sp.set_edgecolor(GRID)
ax2.set_facecolor(PANEL)
ax2.tick_params(colors=TEXT)
ax2.grid(color=GRID, linewidth=0.5, alpha=0.7)

# 3. Confidence distribution
ax3 = fig.add_subplot(gs[2, 0])
ax3.hist(confidence[preds != 0], bins=50, color=ACC, alpha=0.7, density=True, label='Signalled')
ax3.hist(confidence[preds == 0], bins=50, color=GRID, alpha=0.5, density=True, label='Flat')
ax3.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax3.set_xlabel('Confidence (max class prob)')
style(ax3, 'Confidence Distribution')

# 4. Win rate vs confidence
ax4 = fig.add_subplot(gs[2, 1])
threshs = np.arange(0.33, 0.81, 0.01)
wr_curve, n_curve = [], []
for t in threshs:
    mask = (confidence[:-1] >= t) & (preds[:-1] != 0)
    if mask.sum() < 10:
        wr_curve.append(np.nan); n_curve.append(0)
    else:
        tp = trade_pnl[mask]
        wr_curve.append((tp > 0).mean() * 100)
        n_curve.append(mask.sum())
ax4.plot(threshs * 100, wr_curve, color=ACC, linewidth=2)
ax4.axhline(50, color=GRID, linestyle='--', linewidth=0.8)
ax4.axhline(52, color=UP,   linestyle=':',  linewidth=0.8, label='~52% maker b/e')
ax4.set_xlabel('Confidence threshold (%)')
ax4.set_ylabel('Win rate (%)')
ax4.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax4b = ax4.twinx()
ax4b.bar(threshs * 100, n_curve, width=0.8, alpha=0.25, color=GOLD)
ax4b.set_ylabel('# trades', color=GOLD)
ax4b.tick_params(colors=GOLD, labelsize=7)
style(ax4, 'Win Rate vs Confidence Threshold')

fig.suptitle(f'LSTM — threshold=${THRESHOLD:.0f}  horizon={HORIZON} bars ({HORIZON*WINDOW_SEC}s)  seq={SEQ_LEN} bars',
             color=TEXT, fontsize=12, fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_results.png'), dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print(f'Chart saved.')

In [ ]:
# ── Cell 11: Save model ───────────────────────────────────────────────────
import pickle

save_path = os.path.join(OUTPUT_DIR, f'lstm_t{THRESHOLD:.0f}_h{HORIZON}.pt')
torch.save({
    'model_state':  model.state_dict(),
    'scaler':       scaler,
    'features':     features,
    'seq_len':      SEQ_LEN,
    'hidden_dim':   HIDDEN_DIM,
    'num_layers':   NUM_LAYERS,
    'dropout':      DROPOUT,
    'threshold':    THRESHOLD,
    'horizon':      HORIZON,
    'val_acc':      max(val_accs),
    'test_win_rate': win_rate,
    'n_test_trades': n_trades,
}, save_path)
print(f'Model saved → {save_path}')
print(f'\nSummary:')
print(f'  Features      : {len(features)}')
print(f'  Sequence length: {SEQ_LEN} bars ({SEQ_LEN*WINDOW_SEC}s)')
print(f'  Val accuracy  : {max(val_accs)*100:.2f}%')
print(f'  Test win rate : {win_rate*100:.2f}%  ({n_trades:,} trades)')
print(f'  Strategy P&L  : ${strat_pnl:+,.2f}  (zero fees)')
print(f'  After maker   : ${strat_pnl - n_trades*maker_fee:+,.2f}')

In [ ]:
# ── Cell 12: Load saved model (run this to resume without retraining) ─────
#
# checkpoint = torch.load(save_path, map_location=device)
# features   = checkpoint['features']
# scaler     = checkpoint['scaler']
# model      = BtcLSTM(len(features), checkpoint['hidden_dim'],
#                       checkpoint['num_layers'], checkpoint['dropout']).to(device)
# model.load_state_dict(checkpoint['model_state'])
# model.eval()
# print('Model loaded. Val acc:', checkpoint['val_acc']*100, '%')
print('Uncomment the block above to load a saved model.')